In [12]:
import joblib

X_under = joblib.load('../InitData/result/X_under.pkl')


In [13]:
import re
import pandas as pd
from tqdm import tqdm
from pythainlp import word_tokenize
from pythainlp.corpus.common import thai_stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction import text

tqdm.pandas()

# ===============================
# 🔹 1. รวม Stopwords ไทย + อังกฤษ
# ===============================
thai_stopwords_set = set(thai_stopwords())
english_stopwords_set = set(text.ENGLISH_STOP_WORDS)

# เพิ่ม stopwords ไทยเพิ่มเติมที่เจอบ่อยแต่ไม่ช่วยจำแนก
thai_stopwords_set.update([
    'ของ', 'ใน', 'ที่', 'จาก', 'ไป', 'ด้วย', 'กับ', 'และ', 'หรือ', 'แต่', 'เพราะ',
    'ถ้า', 'หาก', 'เขา', 'คุณ', 'ฉัน', 'เรา', 'ทุกคน', 'นี้', 'นั้น', 'คือ', 'การ',
    'ได้', 'มี', 'เป็น', 'ซึ่ง', 'ว่า', 'ก็', 'โดย', 'เช่น', 'เพื่อ'
])

# รวม stopwords ทั้งสองภาษา
all_stopwords = thai_stopwords_set.union(english_stopwords_set)

# ===============================
# 🔹 2. ฟังก์ชันล้างข้อความ
# ===============================
def clean_text(text: str) -> str:
    text = str(text).lower().strip()

    # ลบ URL, mention, hashtag
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r"@\w+|#\w+", "", text)

    # เก็บเฉพาะอักษรไทย อังกฤษ ตัวเลข และช่องว่าง
    text = re.sub(r"[^ก-๙a-zA-Z0-9\s]", " ", text)

    # ตัดช่องว่างเกิน
    text = re.sub(r"\s+", " ", text).strip()

    return text

# ===============================
# 🔹 3. Tokenizer ผสม (ไทย + อังกฤษ)
# ===============================
def hybrid_tokenizer(text: str):
    text = clean_text(text)
    segments = re.findall(r'[a-zA-Z]+|[ก-๙]+|\d+', text)

    tokens = []
    for seg in segments:
        if re.match(r'[ก-๙]+', seg):  # ถ้าเป็นคำไทย
            tokens.extend(word_tokenize(seg, engine='newmm'))
        else:
            tokens.append(seg.lower())

    # ตัด stopwords และคำสั้นเกินไป
    return [w for w in tokens if w not in all_stopwords and len(w) > 1]


# รวม Title + Body
X_under["Text"] = X_under["Title"].fillna("") + " " + X_under["Body"].fillna("")

# ===============================
# 🔹 5. Tokenization + Join String
# ===============================
X_under["Tokens"] = X_under["Text"].progress_apply(hybrid_tokenizer)
X_under["TokenStr"] = X_under["Tokens"].apply(lambda x: " ".join(x))

# ===============================
# 🔹 6. TF-IDF Vectorizer
# ===============================
vectorizer = TfidfVectorizer(
    analyzer="word",
    max_df=0.9,
    min_df=1,
    stop_words=list(all_stopwords),
    ngram_range=(1, 2),
    max_features=10000
)

X_tfidf = vectorizer.fit_transform(X_under["TokenStr"])

# ===============================
# 🔹 7. แสดงผล
# ===============================
print("✅ สร้าง TF-IDF สำเร็จ!")
print("Shape:", X_tfidf.shape)
print("\n🔸 ตัวอย่าง 20 คำแรกใน vocabulary:")
print(vectorizer.get_feature_names_out()[:20])

print("\n🔸 ตัวอย่าง Tokens:")
print(X_under[["Text", "Tokens"]].head())


  0%|          | 0/4245 [00:00<?, ?it/s]

100%|██████████| 4245/4245 [00:17<00:00, 244.70it/s]
C:\Users\SUPHASET\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\feature_extraction\text.py:406: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['กคน', 'กคร', 'กครา', 'กคราว', 'กจะ', 'กช', 'กต', 'กท', 'กทาง', 'กน', 'กระท', 'กระน', 'กระไร', 'กล', 'กว', 'กส', 'กหน', 'กอ', 'กอย', 'กำล', 'กเม', 'กแห', 'กๆ', 'ขณะท', 'ขณะน', 'ขณะหน', 'ขณะเด', 'คงอย', 'คร', 'ครบคร', 'ครบถ', 'คราท', 'คราน', 'คราวก', 'คราวท', 'คราวน', 'คราวหน', 'คราวหล', 'คราวโน', 'คราหน', 'คล', 'งก', 'งกระน', 'งกล', 'งกว', 'งข', 'งคง', 'งคน', 'งครา', 'งคราว', 'งง', 'งจ', 'งจน', 'งจะ', 'งจาก', 'งต', 'งท', 'งน', 'งบ', 'งปวง', 'งมวล', 'งละ', 'งว', 'งส', 'งหน', 'งหมด', 'งหมาย', 'งหล', 'งหลาย', 'งอย', 'งเก', 'งเคย', 'งเน', 'งเป', 'งเม', 'งแก', 'งแต', 'งแม', 'งแล', 'งโง', 'งโน', 'งใด', 'งใหญ', 'งไง', 'งได', 'งไหน', 'งๆ', 'งๆจ', 

✅ สร้าง TF-IDF สำเร็จ!
Shape: (4245, 10000)

🔸 ตัวอย่าง 20 คำแรกใน vocabulary:
['00' '00 01' '00 12' '00 15' '00 16' '00 22' '00 24' '00 นท' '00 ประชาชน'
 '00 เวลา' '000' '000 000' '000 100' '000 300' '000 500' '000 คน' '000 ทธ'
 '000 นผล' '000 บาท' '000 าน']

🔸 ตัวอย่าง Tokens:
                                                 Text  \
0   เกิดแผ่นดินไหว ขนาด 3.6 บริเวณทะเลอ่าวไทย ข้อม...   
2   สูบบุหรี่ไฟฟ้า ทำให้ปอดทะลุ ปอดรั่ว ถึงขั้นหัว...   
4   ธ.ก.ส. ออกสินเชื่อเกษตรวิวัฒน์ ให้กู้สูงสุด 8 ...   
6   กฟผ. ลดค่าล้างแอร์ 200 บาท 15,000 สิทธิ์ รับสิ...   
11  รัฐบาลตั้งศูนย์รับเรื่องร้องทุกข์ แก้ปัญหาใบรั...   

                                               Tokens  
0   [แผ่นดินไหว, ขนาด, บริเวณ, ทะเล, อ่าวไทย, ข้อม...  
2   [สูบบุหรี่, ไฟฟ้า, ปอด, ทะลุ, ปอด, รั่ว, ถึงขั...  
4   [สินเชื่อ, เกษตร, วิวัฒน์, กู้, ล้าน, บาท, หนุ...  
6   [กฟผ, ลด, ค่า, ล้าง, แอร์, 200, บาท, 15, 000, ...  
11  [รัฐบาล, ศูนย์, เรื่อง, ร้องทุกข์, แก้ปัญหา, ใ...  


Testing Result

In [14]:
sample_text = "การเรียนรู้ด้วยตนเองเป็นวิธีที่ดีที่สุดในการพัฒนาทักษะ"
tokens = hybrid_tokenizer(sample_text)

print("Tokens:", tokens)
token_str = ' '.join(tokens)
print("\nToken string for TF-IDF:", token_str)


Tokens: ['การเรียนรู้', 'วิธี', 'ดี', 'การพัฒนา', 'ทักษะ']

Token string for TF-IDF: การเรียนรู้ วิธี ดี การพัฒนา ทักษะ


Save Result

In [15]:
import joblib

# บันทึก TF-IDF vectorizer
joblib.dump(vectorizer, 'result/vectorizer.pkl')

# บันทึก X_tfidf (sparse matrix)
joblib.dump(X_tfidf, 'result/X_tfidf.pkl')

print("✅ บันทึกข้อมูลเรียบร้อยแล้ว!")


✅ บันทึกข้อมูลเรียบร้อยแล้ว!
